In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.model_selection import train_test_split, KFold
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.inspection import PartialDependenceDisplay
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.tree import DecisionTreeRegressor
import matplotlib.font_manager as fm
import seaborn as sns
import os



In [ ]:
#폰트 출력
font_list = [font.name for font in fm.fontManager.ttflist]
font_list

건축경과일 추가 코드

In [ ]:
df = pd.read_csv("seoul_with_city_gu_dong.csv")

In [ ]:
df.isnull().sum()

In [ ]:
missing_city_rows = df[(df['gu'].isna()) | (df['gu'].str.strip() == "")]
print(missing_city_rows)

In [ ]:
missing_dong = df[(df['dong'].isna()) | (df['dong'].str.strip() == "")]
print(missing_dong)

In [ ]:
missing_sales = df[df['avg_sales'].isna()]
print(missing_sales)

In [ ]:

plt.rc("font", family = "Malgun Gothic")
sns.set(font="Malgun Gothic",
rc={"axes.unicode_minus":False}, style='white')
df = pd.read_csv("House.csv")

In [ ]:
df.describe()

In [ ]:
df.boxplot(column=['households'])
plt.show()

In [ ]:
df['buildDate'].hist(bins=70)
plt.show()

In [ ]:
df['m2'].hist(bins=300)
plt.show()

In [ ]:
df.boxplot(column=['p'])
plt.show()

In [ ]:
sns.boxplot(df['avg_sales'])
plt.show()

In [ ]:
drop_df = df.drop(['lat','lng','address','city','gu','dong'], axis=1, inplace = False)
drop_df

In [ ]:
drop_df.corr()

In [ ]:
df_cor = drop_df.corr()
sns.heatmap(df_cor,annot=True, fmt=".2f",
            annot_kws={"size": 9, "weight": "bold"})
plt.show()

In [ ]:
plt.plot('avg_sales', 'm2',
         data=df,
         linestyle='none',
         marker='o',
         markersize=5,
         color='blue',
         alpha=0.5)
plt.title('Scatter Plot')
plt.xlabel('avg_sales')
plt.ylabel('m2')
plt.show()

In [ ]:
plt.plot('avg_sales', 'buildDate',
         data=df,
         linestyle='none',
         marker='o',
         markersize=5,
         color='blue',
         alpha=0.5)
plt.title('Scatter Plot')
plt.xlabel('avg_sales')
plt.ylabel('buildDate')
plt.show()

In [ ]:
plt.plot('avg_sales', 'p',
         data=df,
         linestyle='none',
         marker='o',
         markersize=5,
         color='blue',
         alpha=0.5)
plt.title('Scatter Plot')
plt.xlabel('avg_sales')
plt.ylabel('p')
plt.show()

In [ ]:
print(df['city'].value_counts())
sns.countplot(x = 'city', data = df)
plt.show()

In [ ]:

gu_order = df['gu'].value_counts().index
print(df['dong'].value_counts())
# 시각화
plt.figure(figsize=(10, 6))
sns.countplot(x='gu', data=df, order=gu_order)

plt.xticks(rotation=90)
plt.title('구별 데이터 개수 (빈도순 정렬)')
plt.xlabel('구')
plt.ylabel('개수')
plt.show()

In [ ]:
fig = plt.figure(figsize = (12, 20))
ax = fig.gca()
sns.boxplot(x='avg_sales', y='gu', data=df, ax=ax)
plt.show()

In [ ]:
df.isnull().sum()

In [ ]:
# 각 city별 개수 세기
city_counts = df['city'].value_counts()

print(city_counts)

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:

drop_df = df.drop(['lat','lng','address','city','gu','dong','p'], axis=1, inplace = False)
drop_df

# 숫자형만 남기기
X_vif = drop_df.select_dtypes(include=[np.number]).copy()

# inf → NaN
X_vif = X_vif.replace([np.inf, -np.inf], np.nan)

X_vif = X_vif.dropna()


In [ ]:


X = X_vif  # 수치형 변수만 넣기
X = add_constant(X)

vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif)


In [ ]:


fig, ax = plt.subplots(figsize=(6, len(vif)*0.5 + 1))
ax.axis('off')

table = ax.table(
    cellText=vif.values,
    colLabels=vif.columns,
    cellLoc='center',
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.3)

plt.savefig("vif_table.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:


X = df[['m2','p']].copy()  
X = add_constant(X)

vif_df = pd.DataFrame()
vif_df["feature"] = X.columns
vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_df)


In [ ]:
df[['m2','p']].head(20)
df[['m2','p']].sample(20)
df[['m2','p']].corr()


In [ ]:
df.isnull().sum()

<h1>모델링 모델링 모델링 모델링 모델링<h1>

<h1>모델링 모델링 모델링 모델링 모델링<h1>

<h1>모델링 모델링 모델링 모델링 모델링<h1>

In [ ]:





# 데이터 로드
df = pd.read_csv("BaseData.csv")

# 쓸 피처만 선택 
base_features = [
    'lat', 'lng',
    'households', 'buildDate', 'score', 'm2',
    'city', 'gu', 'dong'
]

X = df[base_features].copy()
y = df['avg_sales'].copy()

# 문자열 정리 + 결측치 채우기
for col in ['city', 'gu', 'dong']:
    X[col] = (
        X[col]
        .astype(str)
        .str.strip()
        .replace({'nan': None})
        .fillna('Unknown')
    )

# 범주형, 수치형 지정
cat_features = ['city', 'gu', 'dong']
num_features = [c for c in X.columns if c not in cat_features]


# -----------------------------
# 10-Fold Cross-Validation
# -----------------------------
kf = KFold(n_splits=10, shuffle=True, random_state=42)

rmse_list = []
r2_list = []
mae_list = []
mape_list = []


fold = 1
for train_idx, valid_idx in kf.split(X):
    X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    valid_pool = Pool(X_val, y_val, cat_features=cat_features)

    cv_model = CatBoostRegressor(
        loss_function='RMSE',
        eval_metric='RMSE',
        depth=6,
        learning_rate=0.05,
        l2_leaf_reg=1,
        bagging_temperature=0,
        iterations=1200,
        random_state=42,
        thread_count=-1,
        verbose=False,
    )

    cv_model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    y_val_pred = cv_model.predict(valid_pool)

    rmse_fold = np.sqrt(mean_squared_error(y_val, y_val_pred))
    mae_fold  = mean_absolute_error(y_val, y_val_pred)

    mask_nonzero = y_val != 0
    mape_fold = np.mean(np.abs((y_val[mask_nonzero] - y_val_pred[mask_nonzero]) / y_val[mask_nonzero])) * 100

    r2_fold   = r2_score(y_val, y_val_pred)

    rmse_list.append(rmse_fold)
    mae_list.append(mae_fold)
    mape_list.append(mape_fold)
    r2_list.append(r2_fold)

    print(f"[Fold {fold}] RMSE: {rmse_fold:.3f}, MAE: {mae_fold:.3f}, MAPE: {mape_fold:.3f}%, R²: {r2_fold:.3f}")

    fold += 1

print("\n===== [베이스 CatBoost] 10-Fold Cross-Validation 결과 =====")
print(f"평균 RMSE: {np.mean(rmse_list):.3f} ± {np.std(rmse_list):.3f}")
print(f"평균 MAE : {np.mean(mae_list):.3f} ± {np.std(mae_list):.3f}")
print(f"평균 MAPE: {np.mean(mape_list):.3f}% ± {np.std(mape_list):.3f}")
print(f"평균 R²  : {np.mean(r2_list):.3f} ± {np.std(r2_list):.3f}")


# ======================================
# XAI: PDP, Proxy model, LIME, SHAP
# ======================================


# 전체 데이터를 이용해 최종 모델 학습
full_pool = Pool(X, y, cat_features=cat_features)

model = CatBoostRegressor(
    loss_function='RMSE',
    eval_metric='RMSE',
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=1,
    bagging_temperature=0,
    iterations=1200,
    random_state=42,
    thread_count=-1,
    verbose=100,
)

model.fit(full_pool)  # eval_set 없이 전체 데이터로 최종 학습



# Feature Importance
fi = model.get_feature_importance(train_pool)
fi_df = pd.DataFrame({'feature': X.columns, 'importance': fi}).sort_values('importance', ascending=False)
print("\n=== Feature Importance ===")
print(fi_df)



# Partial Dependence Plot (PDP)
#    - 베이스 모델에서 중요한 연속형 피처 위주로
PartialDependenceDisplay.from_estimator(
    model,
    X,  # 전체 데이터
    features=['m2', 'buildDate', 'lat', 'lng'],
    grid_resolution=50
)

# Proxy model (대리 모델: CatBoost → 작은 의사결정나무로 근사)
#    - CatBoost 예측값(y_hat)을 타깃으로,
#      수치형 피처(num_features)만 사용해 간단한 트리 학습
X_num_full = X[num_features]
y_hat_full = model.predict(X)

proxy = DecisionTreeRegressor(max_depth=3, random_state=42)
proxy.fit(X_num_full, y_hat_full)

print("\n=== Proxy Model Feature Importance (DecisionTree, depth=3) ===")
print(
    pd.Series(proxy.feature_importances_, index=num_features)
      .sort_values(ascending=False)
)


# SHAP 
shap.initjs()

shap_explainer = shap.TreeExplainer(model)
# 계산량 줄이려고 X_test 기준으로 예시
shap_values = shap_explainer.shap_values(X)

# 전체 특성 중요도 (bar summary)
shap.summary_plot(shap_values, X, plot_type='bar')

# 분포
shap.summary_plot(shap_values, X)




<h1>2모델링 모델링 모델링 모델링 모델링2<h1>

#10-fold버전

In [ ]:
df = pd.read_csv('useData2.csv')
drop_df = df.drop(['lat','lng','address','city','gu','dong'], axis=1, inplace = False)
drop_df

# 1) 숫자형만 남기기
X_vif = drop_df.select_dtypes(include=[np.number]).copy()

# 2) inf → NaN
X_vif = X_vif.replace([np.inf, -np.inf], np.nan)

# 3) NaN 제거
X_vif = X_vif.dropna()

X = X_vif  # 수치형 변수만 넣기
X = add_constant(X)

vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

fig, ax = plt.subplots(figsize=(6, len(vif)*0.5 + 1))
ax.axis('off')

table = ax.table(
    cellText=vif.values,
    colLabels=vif.columns,
    cellLoc='center',
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.3)

plt.savefig("vif_table.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:

# 데이터 로드
df = pd.read_csv("useData2.csv")

# 피처 목록 정의
features = [
    'lat', 'lng',
    'households', 'buildDate', 'score', 'm2',
    'school_total', 'subway_total', 'public_total',
    'hospital_total', 'mart_total',
    'city', 'gu', 'dong',
    'subway_nearest_km', 'dong_te',
    'household_growth_2021', 'apt_transactions_2021'
]

X = df[features].copy()
y = df['avg_sales'].copy()

# 문자열 정리 + 결측치 처리
for col in ['city', 'gu']:
    X[col] = (
        X[col]
        .astype(str)
        .str.strip()
        .replace({'nan': None})
        .fillna('Unknown')
    )

# 범주형, 수치형 지정
cat_features = ['city', 'gu', 'dong']
num_features = [c for c in X.columns if c not in cat_features]





# --------------------------------
# 10-Fold Cross-Validation 추가
# --------------------------------

kf = KFold(n_splits=10, shuffle=True, random_state=42)

rmse_list = []
mae_list = []
mape_list = []
r2_list = []


fold = 1
for train_idx, valid_idx in kf.split(X):
    X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    valid_pool = Pool(X_val, y_val, cat_features=cat_features)

    cv_model = CatBoostRegressor(
        loss_function='RMSE',
        eval_metric='RMSE',
        depth=8,                
        learning_rate=0.1,      
        l2_leaf_reg=3,          
        bagging_temperature=0,  
        iterations=1200,        
        random_state=42,
        thread_count=-1,
        verbose=False,
    )


    cv_model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    y_val_pred = cv_model.predict(valid_pool)

    rmse_fold = np.sqrt(mean_squared_error(y_val, y_val_pred))
    mae_fold  = mean_absolute_error(y_val, y_val_pred)

    mask_nonzero = y_val != 0
    mape_fold = np.mean(np.abs((y_val[mask_nonzero] - y_val_pred[mask_nonzero]) / y_val[mask_nonzero])) * 100

    r2_fold   = r2_score(y_val, y_val_pred)

    rmse_list.append(rmse_fold)
    mae_list.append(mae_fold)
    mape_list.append(mape_fold)
    r2_list.append(r2_fold)

    print(f"[Fold {fold}] RMSE: {rmse_fold:.3f}, MAE: {mae_fold:.3f}, MAPE: {mape_fold:.3f}%, R²: {r2_fold:.3f}")

    fold += 1

print("\n===== 10-Fold Cross-Validation 평균 결과 =====")
print(f"평균 RMSE : {np.mean(rmse_list):.3f} ± {np.std(rmse_list):.3f}")
print(f"평균 MAE  : {np.mean(mae_list):.3f} ± {np.std(mae_list):.3f}")
print(f"평균 MAPE : {np.mean(mape_list):.3f}% ± {np.std(mape_list):.3f}")
print(f"평균 R²   : {np.mean(r2_list):.3f} ± {np.std(r2_list):.3f}")

# ======================================
#  추가 모델 해석(Explainability)
# ======================================


# 전체 데이터를 이용해 최종 모델 학습
full_pool = Pool(X, y, cat_features=cat_features)

model = CatBoostRegressor(
    loss_function='RMSE',
    eval_metric='RMSE',
    depth=8,                
    learning_rate=0.1,      
    l2_leaf_reg=3,         
    bagging_temperature=0,  
    iterations=1200,       
    random_state=42,
    thread_count=-1,
    verbose=100,
)


model.fit(full_pool)  # eval_set 없이 전체 데이터로 최종 학습


# Feature Importance
fi = model.get_feature_importance(train_pool)
fi_df = pd.DataFrame({'feature': X.columns, 'importance': fi}).sort_values('importance', ascending=False)
print("\n=== Feature Importance ===")
print(fi_df)

# ======================================
# XAI: PDP, Proxy model, LIME, SHAP
# ======================================

# Partial Dependence Plot (PDP)
#   
PartialDependenceDisplay.from_estimator(
    model,
    X,  # 전체 데이터
    features=['m2', 'buildDate', 'households', 'dong_te'],
    grid_resolution=50
)



# 2) Proxy model 
X_num_full = X[num_features]          # 수치형 피처만 사용
y_hat_full = model.predict(X)        # CatBoost 예측값

proxy = DecisionTreeRegressor(max_depth=3, random_state=42)
proxy.fit(X_num_full, y_hat_full)

print("\n=== Proxy Model Feature Importance (DecisionTree, depth=3) ===")
print(
    pd.Series(proxy.feature_importances_, index=num_features)
      .sort_values(ascending=False)
)


# 4) SHAP 
shap.initjs()

shap_explainer = shap.TreeExplainer(model)
# 계산량 줄이려고 X_test 기준으로 예시
shap_values = shap_explainer.shap_values(X)

# 전체 특성 중요도 (bar summary)
shap.summary_plot(shap_values, X, plot_type='bar')

# 분포
shap.summary_plot(shap_values, X)

